In [1]:
import math, torch
from rdkit import Chem
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from rdkit import RDLogger
lg = RDLogger.logger()
lg.setLevel(RDLogger.CRITICAL)

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.manual_seed(42)

tokenizer = AutoTokenizer.from_pretrained("sagawa/CompoundT5")
model = AutoModelForSeq2SeqLM.from_pretrained("sagawa/CompoundT5")

train_enc = torch.load("../tokenizer/pretraining/train_enc_1M.pt", weights_only=False)
valid_enc = torch.load("../tokenizer/pretraining/valid_enc.pt", weights_only=False)
test_enc  = torch.load("../tokenizer/pretraining/test_enc.pt",  weights_only=False)

class TokenizedDictDataset(Dataset):
    def __init__(self, encoded_dict):
        self.encoded_dict = encoded_dict
        self.length = self.encoded_dict["input_ids"].size(0)
    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encoded_dict.items()}

train_ds = TokenizedDictDataset(train_enc)
valid_ds = TokenizedDictDataset(valid_enc)
test_ds  = TokenizedDictDataset(test_enc)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest")

model.config.use_cache = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

args = TrainingArguments(
    output_dir="Frag2OPST5_1M",
    overwrite_output_dir=True,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,

    learning_rate=5e-4,
    num_train_epochs=4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    seed=42,

    logging_steps=1000,
    eval_strategy="steps",
    eval_steps=10000,
    save_strategy="steps",
    save_steps=10000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    bf16=True,
    fp16=False,

    dataloader_num_workers=0,
    no_cuda=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
    #callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=5e-4)]
)

trainer.train()

trainer.save_model("Frag2OPST5_1M/model")
tokenizer.save_pretrained("Frag2OPST5_1M/tokenizer")

C:\Users\noton\AppData\Local\Temp\ipykernel_13156\2482949230.py:62: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\transformers\data\data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss,Validation Loss
10000,0.104700,0.092281
20000,0.080100,0.074029
30000,0.069900,0.065314
40000,0.059700,0.059597
50000,0.048600,0.055666
60000,0.046000,0.052338


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


('Frag2OPST5_1M/tokenizer\\tokenizer_config.json',
 'Frag2OPST5_1M/tokenizer\\special_tokens_map.json',
 'Frag2OPST5_1M/tokenizer\\tokenizer.json')